In [0]:
%sql
CREATE CATALOG IF NOT EXISTS github_health;

CREATE SCHEMA IF NOT EXISTS github_health.raw;

CREATE VOLUME IF NOT EXISTS github_health.raw.bronze_data;

In [0]:
# Databricks notebook source
# ============================================================
# GitHub Repository Health Analytics
# FULL HISTORICAL LOAD -> DATABRICKS VOLUME (BRONZE)
#
# Repositories:
#   1. apache/spark
#   2. pandas-dev/pandas
#   3. scikit-learn/scikit-learn
#
# Entities:
#   - repo_metadata
#   - commits
#   - issues
#   - pulls
#   - contributors
#
# Output:
# /Volumes/<catalog>/<schema>/<volume>/bronze/
# ============================================================

import requests
import json
import os
import time
import uuid

from datetime import datetime, timedelta, timezone
from pathlib import Path


# ============================================================
# 1. CONFIGURATION
# ============================================================

REPOS = [
    "apache/spark",
    "pandas-dev/pandas",
    "scikit-learn/scikit-learn"
]

API = "https://api.github.com"

# ------------------------------------------------------------
# CHANGE THESE 3 VALUES IF YOUR CATALOG / SCHEMA / VOLUME
# HAVE DIFFERENT NAMES
# ------------------------------------------------------------

CATALOG = "github_health"
SCHEMA = "raw"
VOLUME = "bronze_data"

VOLUME_ROOT = Path(
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
)

BRONZE_ROOT = VOLUME_ROOT / "bronze"


# ============================================================
# 2. FULL LOAD WINDOW
# ============================================================
# One fixed 365-day window is created ONCE.
# All three repositories therefore use exactly the same dates.
# ============================================================

END = datetime.now(timezone.utc).replace(microsecond=0)
START = END - timedelta(days=365)

START_ISO = START.isoformat().replace("+00:00", "Z")
END_ISO = END.isoformat().replace("+00:00", "Z")

RUN_DATE = END.strftime("%Y-%m-%d")
BATCH_ID = str(uuid.uuid4())


print("=" * 80)
print("GITHUB REPOSITORY HEALTH ANALYTICS - FULL LOAD")
print("=" * 80)

print(f"Batch ID       : {BATCH_ID}")
print(f"Start          : {START_ISO}")
print(f"End            : {END_ISO}")
print(f"Volume         : {VOLUME_ROOT}")
print(f"Bronze path    : {BRONZE_ROOT}")

print("=" * 80)


# ============================================================
# 3. GITHUB TOKEN
# ============================================================
#
# Recommended:
# Store the token in a Databricks secret and use:
#
# TOKEN = dbutils.secrets.get(
#     scope="github",
#     key="github-token"
# )
#
# For a quick classroom/test setup, you can temporarily use:
#
# TOKEN = "github_pat_XXXXXXXXXXXXXXXX"
#
# NEVER commit the token to GitHub.
# ============================================================

try:
    TOKEN = dbutils.secrets.get(
        scope="github",
        key="token"
    )
except Exception:
    TOKEN = os.environ.get("GITHUB_TOKEN")


if not TOKEN:
    raise ValueError(
        """
GitHub token not found.

Set either:

1. Databricks secret:
   scope = github
   key   = github-token

OR

2. Environment variable:
   GITHUB_TOKEN
"""
    )


# ============================================================
# 4. CREATE GITHUB SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
})


# ============================================================
# 5. CREATE BRONZE DIRECTORY
# ============================================================

BRONZE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 6. SIZE FORMATTER
# ============================================================

def human_size(num_bytes):
    """
    Convert bytes into KB / MB / GB.
    """

    if num_bytes < 1024:
        return f"{num_bytes} B"

    elif num_bytes < 1024 ** 2:
        return f"{num_bytes / 1024:.2f} KB"

    elif num_bytes < 1024 ** 3:
        return f"{num_bytes / (1024 ** 2):.2f} MB"

    else:
        return f"{num_bytes / (1024 ** 3):.2f} GB"


# ============================================================
# 7. API GET WITH RETRIES + RATE LIMIT HANDLING
# ============================================================

API_REQUESTS = 0


def github_get(url, params=None, retries=5):

    global API_REQUESTS

    for attempt in range(retries):

        response = session.get(
            url,
            params=params,
            timeout=60
        )

        API_REQUESTS += 1

        # Successful request
        if response.status_code == 200:
            return response

        # ----------------------------------------------------
        # RATE LIMIT
        # ----------------------------------------------------

        if response.status_code in (403, 429):

            remaining = response.headers.get(
                "X-RateLimit-Remaining"
            )

            if remaining == "0":

                reset_time = int(
                    response.headers.get(
                        "X-RateLimit-Reset",
                        0
                    )
                )

                wait_seconds = max(
                    reset_time - int(time.time()),
                    0
                ) + 5

                print(
                    f"GitHub rate limit reached."
                    f" Waiting {wait_seconds} seconds..."
                )

                time.sleep(wait_seconds)
                continue

        # ----------------------------------------------------
        # SERVER ERROR -> EXPONENTIAL BACKOFF
        # ----------------------------------------------------

        if response.status_code >= 500:

            wait_seconds = 2 ** attempt

            print(
                f"GitHub server error "
                f"{response.status_code}. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)
            continue

        # Other HTTP errors
        response.raise_for_status()

    raise RuntimeError(
        f"GitHub request failed after "
        f"{retries} attempts: {url}"
    )


# ============================================================
# 8. PAGINATION
# ============================================================

def paginate(url, params=None):
    """
    Generator that follows GitHub's Link rel=next pagination.

    GitHub allows up to 100 records per page.
    """

    if params is None:
        params = {}

    params = {
        **params,
        "per_page": 100
    }

    while url:

        response = github_get(
            url,
            params=params
        )

        data = response.json()

        for record in data:
            yield record

        next_url = response.links.get(
            "next",
            {}
        ).get("url")

        url = next_url

        # next_url already contains its query parameters
        params = None


# ============================================================
# 9. OUTPUT FILE PATH
# ============================================================

def get_output_path(entity, repo):

    repo_folder = repo.replace("/", "__")

    directory = (
        BRONZE_ROOT
        / entity
        / repo_folder
    )

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

    filename = (
        f"{entity}_full_{RUN_DATE}_{BATCH_ID}.jsonl"
    )

    return directory / filename


# ============================================================
# 10. ADD BRONZE INGESTION METADATA
# ============================================================

def prepare_bronze_record(
    record,
    repo,
    endpoint
):

    # Make a copy so the original API object
    # is not modified unexpectedly.
    bronze_record = dict(record)

    bronze_record["_repo"] = repo
    bronze_record["_ingested_at"] = END_ISO
    bronze_record["_load_type"] = "FULL"
    bronze_record["_source_endpoint"] = endpoint
    bronze_record["_batch_id"] = BATCH_ID

    return bronze_record


# ============================================================
# 11. WRITE ONE JSON RECORD
# ============================================================

def write_json_record(
    file_handle,
    record
):
    """
    Writes exactly one JSON object per line.

    Returns number of UTF-8 bytes written.
    """

    line = (
        json.dumps(
            record,
            ensure_ascii=False
        )
        + "\n"
    )

    file_handle.write(line)

    return len(
        line.encode("utf-8")
    )


# ============================================================
# 12. REPOSITORY METADATA
# ============================================================

def load_repo_metadata(repo):

    endpoint = f"{API}/repos/{repo}"

    output_path = get_output_path(
        "repo_metadata",
        repo
    )

    bytes_written = 0

    response = github_get(endpoint)

    record = prepare_bronze_record(
        response.json(),
        repo,
        endpoint
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        bytes_written += write_json_record(
            f,
            record
        )

    return 1, bytes_written, output_path


# ============================================================
# 13. COMMITS
# ============================================================

def load_commits(repo):

    endpoint = (
        f"{API}/repos/{repo}/commits"
    )

    output_path = get_output_path(
        "commits",
        repo
    )

    count = 0
    bytes_written = 0

    print(
        f"      Fetching commits "
        f"{START_ISO} -> {END_ISO}"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        for record in paginate(
            endpoint,
            {
                "since": START_ISO,
                "until": END_ISO
            }
        ):

            bronze_record = prepare_bronze_record(
                record,
                repo,
                endpoint
            )

            bytes_written += write_json_record(
                f,
                bronze_record
            )

            count += 1

            # Progress display every 1000 records
            if count % 1000 == 0:
                print(
                    f"         {count:,} records | "
                    f"{human_size(bytes_written)}"
                )

    return count, bytes_written, output_path


# ============================================================
# 14. ISSUES
# ============================================================

def load_issues(repo):

    endpoint = (
        f"{API}/repos/{repo}/issues"
    )

    output_path = get_output_path(
        "issues",
        repo
    )

    count = 0
    bytes_written = 0

    print(
        f"      Fetching issues updated since "
        f"{START_ISO}"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        for record in paginate(
            endpoint,
            {
                "state": "all",
                "since": START_ISO,
                "sort": "updated",
                "direction": "desc"
            }
        ):

            # ------------------------------------------------
            # GitHub /issues endpoint ALSO returns PR objects.
            #
            # We remove them from the issue dataset because
            # PRs are collected separately.
            # ------------------------------------------------

            if "pull_request" in record:
                continue

            bronze_record = prepare_bronze_record(
                record,
                repo,
                endpoint
            )

            bytes_written += write_json_record(
                f,
                bronze_record
            )

            count += 1

            if count % 1000 == 0:
                print(
                    f"         {count:,} records | "
                    f"{human_size(bytes_written)}"
                )

    return count, bytes_written, output_path


# ============================================================
# 15. PULL REQUESTS
# ============================================================

def load_pulls(repo):

    endpoint = (
        f"{API}/repos/{repo}/pulls"
    )

    output_path = get_output_path(
        "pulls",
        repo
    )

    count = 0
    bytes_written = 0

    print(
        f"      Fetching PRs updated within "
        f"full-load window"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        for record in paginate(
            endpoint,
            {
                "state": "all",
                "sort": "updated",
                "direction": "desc"
            }
        ):

            updated_at = record.get(
                "updated_at"
            )

            # Pull API has no "since" parameter.
            # Because results are newest -> oldest,
            # stop once records are older than START.
            if (
                updated_at
                and updated_at < START_ISO
            ):
                break

            bronze_record = prepare_bronze_record(
                record,
                repo,
                endpoint
            )

            bytes_written += write_json_record(
                f,
                bronze_record
            )

            count += 1

            if count % 1000 == 0:
                print(
                    f"         {count:,} records | "
                    f"{human_size(bytes_written)}"
                )

    return count, bytes_written, output_path


# ============================================================
# 16. CONTRIBUTORS
# ============================================================

def load_contributors(repo):

    endpoint = (
        f"{API}/repos/{repo}/contributors"
    )

    output_path = get_output_path(
        "contributors",
        repo
    )

    count = 0
    bytes_written = 0

    print(
        "      Fetching contributor snapshot"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        for record in paginate(
            endpoint,
            {}
        ):

            bronze_record = prepare_bronze_record(
                record,
                repo,
                endpoint
            )

            bytes_written += write_json_record(
                f,
                bronze_record
            )

            count += 1

            if count % 1000 == 0:
                print(
                    f"         {count:,} records | "
                    f"{human_size(bytes_written)}"
                )

    return count, bytes_written, output_path


# ============================================================
# 17. ENTITY FUNCTIONS
# ============================================================

ENTITIES = {
    "repo_metadata": load_repo_metadata,
    "commits": load_commits,
    "issues": load_issues,
    "pulls": load_pulls,
    "contributors": load_contributors
}


# ============================================================
# 18. FULL LOAD
# ============================================================

summary = []

repository_sizes = {}

grand_total_bytes = 0
grand_total_records = 0


for repo_number, repo in enumerate(
    REPOS,
    start=1
):

    print()
    print("=" * 80)

    print(
        f"REPOSITORY {repo_number}/{len(REPOS)}: "
        f"{repo}"
    )

    print("=" * 80)

    repo_bytes = 0
    repo_records = 0

    repo_start_time = time.time()

    # --------------------------------------------------------
    # LOAD EVERY ENTITY
    # --------------------------------------------------------

    for entity, loader in ENTITIES.items():

        print()
        print(f"   [{entity}]")

        entity_start = time.time()

        count, size_bytes, path = loader(
            repo
        )

        elapsed = (
            time.time() - entity_start
        )

        repo_bytes += size_bytes
        repo_records += count

        grand_total_bytes += size_bytes
        grand_total_records += count

        summary.append({
            "repository": repo,
            "entity": entity,
            "records": count,
            "bytes": size_bytes,
            "size": human_size(size_bytes),
            "path": str(path)
        })

        print(
            f"      DONE"
        )

        print(
            f"      Records : {count:,}"
        )

        print(
            f"      Size    : "
            f"{human_size(size_bytes)}"
        )

        print(
            f"      Time    : "
            f"{elapsed:.1f} seconds"
        )

        print(
            f"      File    : {path}"
        )

        print(
            f"      Repo running total: "
            f"{human_size(repo_bytes)}"
        )

    repo_elapsed = (
        time.time() - repo_start_time
    )

    repository_sizes[repo] = repo_bytes

    # --------------------------------------------------------
    # REPOSITORY SUMMARY
    # --------------------------------------------------------

    print()
    print("-" * 80)

    print(
        f"FINISHED REPOSITORY: {repo}"
    )

    print(
        f"Records loaded : "
        f"{repo_records:,}"
    )

    print(
        f"Repository size: "
        f"{human_size(repo_bytes)}"
    )

    print(
        f"Elapsed time   : "
        f"{repo_elapsed:.1f} seconds"
    )

    print("-" * 80)


# ============================================================
# 19. SAVE WATERMARKS
# ============================================================

watermarks = {}

for repo in REPOS:

    for entity in ENTITIES:

        watermarks[
            f"{repo}|{entity}"
        ] = END_ISO


watermark_directory = (
    BRONZE_ROOT
    / "_watermarks"
)

watermark_directory.mkdir(
    parents=True,
    exist_ok=True
)

watermark_path = (
    watermark_directory
    / "watermarks.json"
)

with open(
    watermark_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        watermarks,
        f,
        indent=2
    )


# ============================================================
# 20. SAVE FULL LOAD MANIFEST
# ============================================================

manifest_directory = (
    BRONZE_ROOT
    / "_manifests"
)

manifest_directory.mkdir(
    parents=True,
    exist_ok=True
)

manifest_path = (
    manifest_directory
    / f"full_load_{RUN_DATE}_{BATCH_ID}.json"
)


manifest = {

    "batch_id": BATCH_ID,

    "load_type": "FULL",

    "window_start": START_ISO,

    "window_end": END_ISO,

    "run_date": RUN_DATE,

    "repositories": REPOS,

    "entities": list(
        ENTITIES.keys()
    ),

    "repository_sizes_bytes":
        repository_sizes,

    "total_records":
        grand_total_records,

    "total_bytes":
        grand_total_bytes,

    "total_human_size":
        human_size(grand_total_bytes),

    "api_requests":
        API_REQUESTS,

    "files":
        summary
}


with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ============================================================
# 21. FINAL FULL LOAD REPORT
# ============================================================

print()
print()
print("=" * 80)

print(
    "FULL HISTORICAL LOAD COMPLETE"
)

print("=" * 80)

print(
    f"Batch ID : {BATCH_ID}"
)

print(
    f"Window   : "
    f"{START_ISO} -> {END_ISO}"
)

print()

print("SIZE BY REPOSITORY")

print("-" * 80)

for repo in REPOS:

    print(
        f"{repo:32s} "
        f"{human_size(repository_sizes[repo]):>15s}"
    )


print("-" * 80)

print(
    f"{'TOTAL FULL LOAD':32s} "
    f"{human_size(grand_total_bytes):>15s}"
)

print()

print(
    f"Total records      : "
    f"{grand_total_records:,}"
)

print(
    f"Total API requests : "
    f"{API_REQUESTS:,}"
)

print(
    f"Watermarks         : "
    f"{watermark_path}"
)

print(
    f"Manifest           : "
    f"{manifest_path}"
)

print(
    f"Bronze root        : "
    f"{BRONZE_ROOT}"
)

print("=" * 80)